In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_postgres.vectorstores import PGVector


CONNECTION = os.environ.get("PGVT_CONNECTION")

COLLECTION_NAME = "documents"
DOCUMENT_RELEVANT_THRESHOLD = 0.7
RETRY_RETRIEVAL_COUNT = 1


llm = ChatNVIDIA(
    model="meta/llama-3.1-405b-instruct",
    temperature=0.0,
)

chat_model = ChatNVIDIA(
    model="meta/llama-3.1-405b-instruct",
    temperature=0.5,
)

ranking = NVIDIARerank(
    model="nvidia/nv-rerankqa-mistral-4b-v3",
    truncate="END"
)

embeddings = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-mistral-7b-v2",
    truncate="END"
)

vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=CONNECTION,
    use_jsonb=True,
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

class RouteQuery(BaseModel):
    """
    A model representing a routing decision for user queries. 
    Determines whether a query is processed via RAG (Retrieval-Augmented Generation) 
    or handled as a regular chatbot conversation.
    """

    route: Literal["RAG", "Chatbot"] = Field(
        ..., description="The determined route for processing: 'RAG' or 'Chatbot'.")


structured_llm_router = llm.with_structured_output(RouteQuery)
system = """
You are a smart router determining whether to process a user's input using the RAG retrieval system or 
handle it as a conversational response from the chatbot. Follow these steps:

  1. Use the RAG process if the input. 
     * Requires detailed factual retrieval from a specific knowledge base or document.
     * Mentions topics not covered by the chatbot's general knowledge or external tools

     Examples include:
      * 'What is the financial projection for Q3 2024?'
      * 'Summarize the company's annual report.'
      * 'Retrieve details about document X.'

  2. Handle it as a chatbot interaction if the input. 
     * Seeks up-to-date information such as weather, current events, or trending topics (use the web search tool as needed).
     * Relates to user-specific references that can be resolved using long-term memory.
     * Involves casual conversation, creative tasks, or opinion-based queries

     Examples include:
      * 'What's the weather like in DaNang today?'
      * 'Tell me a joke.'
      * 'Hi'

Output:
  * If RAG is needed, respond with: "RAG"
  * If it's a regular conversation, respond directly as a "Chatbot"
"""

route_prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{question}")])

route_chain = route_prompt | structured_llm_router

In [ ]:
from typing import List
from langgraph.graph import MessagesState
from typing_extensions import TypedDict
from langchain.schema import Document


class State(MessagesState):
    question: str
    generation: str
    documents: List[Document]
    retry_count: int

In [ ]:
### Search

from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(k=3)

In [ ]:
### Utils def

# Util def to get all documents content
def format_docs(docs: List[Document]):
    return "\n\n".join(doc.page_content for doc in docs)

# Get user_id from config
from langchain_core.runnables import RunnableConfig
def get_user_id(config: RunnableConfig) -> str:
    user_id = config["configurable"].get("user_id", "")
    if user_id is None:
        raise ValueError("User ID needs to be provided to save a memory.")

    return user_id

# Sigmoid activation function
import numpy as np

def sigmoid(x: float):
    return 1 / (1 + np.exp(-x))

In [ ]:
import numpy as np
def sigmoid(x: float):
    return 1 / (1 + np.exp(-x))
sigmoid(-0.6796875)


In [ ]:
### Define Graph Nodes
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from copilotkit.langchain import copilotkit_customize_config

def retrieve(state: State, config: RunnableConfig) -> State:
    """
    Retrieve documents

    Args:
        state (dict): The current graph state
        config (RunnableConfig): The runtime configuration for the agent.

    Returns:
        state (dict): New key added to state, documents, that contains retrieved documents
    """
    retry_count = state.get("retry_count", -1)
    last_message = state["messages"][-1]

    if retry_count >= RETRY_RETRIEVAL_COUNT:
        return {"question": last_message.content, "retry_count": retry_count + 1}

    user_id = get_user_id(config)
    filter = {"user_id": {"$eq": user_id}}
    search_kwargs = {
        "k": 3,
        "fetch_k": 5,
        # "filter": filter
    }
    retriever = vector_store.as_retriever(search_type="mmr",
                                          search_kwargs=search_kwargs)
    documents = retriever.invoke(last_message.content)

    return {"documents": documents, "retry_count": retry_count + 1, "question": last_message.content}


tools = [web_search_tool]

def agent(state: State) -> State:
    """
    Invokes the agent model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply end.

    Args:
        state (messages): The current state

    Returns:
        dict: The updated state with the agent response appended to messages
    """
    messages = state["messages"]
    context = state.get("generation", "No context provided")
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", """You are given a Context that provides background information or details related to
                      a specific topic, scenario, or situation. Based on this provided context, 
                      you are asked to answer user question in a clear, comprehensive, and accurate manner.
                      Your answer should take into account the information in the context, ensuring 
                      that it is relevant and well-supported.
                      Context: {context}"""),
        ("placeholder", "{messages}")             
    ])
    model = chat_model.bind_tools(tools)
    chain = prompt_template | model
    response = chain.invoke({"context": context, "messages": messages})

    return {"messages": [response]}


def rewrite(state: State, config: RunnableConfig) -> State:
    """
    Transform the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased question
    """
    question = state["question"]
    human_msg = [
        ("human", """You a question re-writer that converts an input question to a better version that is optimized,
        for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning.
        Here is the information you need to know:
        INIT QUESTION: {question}
        IMPROVED QUESTION:

        CAVEAT: Only give back the final question
        """)
        ]
    prompt = ChatPromptTemplate.from_messages(human_msg) | chat_model | StrOutputParser()
    modified_config = copilotkit_customize_config(config, emit_messages=False)
    improved_question = prompt.invoke({"question": question}, config=modified_config)

    return {"question": improved_question}


def generate(state: State, config: RunnableConfig) -> State: 
    """
    Generate answer

    Args:
        state (dict): The current graph state
        config (RunnableConfig): Config pass through graph

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    question = state["question"]
    docs = state["documents"]

    rag_prompt = """You are an assistant for question-answering tasks. 
              Use the following pieces of retrieved context to answer the question. 
              If you don't know the answer, just say that you don't know. 
              Use three sentences maximum and keep the answer concise.
              Question: <question>\n{question}\n</question>
              Context: <context>\n{context}\n</context>
              Answer: 
              """
    generate_prompt = ChatPromptTemplate.from_template(rag_prompt)
    rag_chain = generate_prompt | chat_model | StrOutputParser()
    modified_config = copilotkit_customize_config(config, emit_messages=False)
    generation = rag_chain.invoke({"context": format_docs(docs), "question":  question}, config=modified_config)

    return {"generation": generation}


In [ ]:
### Edges

def grade_documents(state: State) -> Literal["transform", "unavailable", "available"]:
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (dict): The current graph state

    Returns:
        str: A decision for whether the documents are relevant or not
    """
    messages = state["messages"]
    question = messages[0].content
    document = state["documents"]
    retry_count = state["retry_count"]

    # Using ranking model to filter irrelevant documents
    ranking_docs = ranking.compress_documents(
      query=question,
      documents=document,
    )
    ranking_docs = [doc for doc in ranking_docs if sigmoid(doc.metadata.get("relevance_score", 0.0)) >= DOCUMENT_RELEVANT_THRESHOLD]

    #  if not ranking_docs:
    #     return "transform"
    # elif retry_count < RETRY_RETRIEVAL_COUNT:
    #     return "unavailable"

    if retry_count > RETRY_RETRIEVAL_COUNT:
        return "unavailable"
    elif not ranking_docs:
        return "transform"
    else:
        return "available"



In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

workflow = StateGraph(State)

# Define the nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)
workflow.add_node("rewrite", rewrite)
workflow.add_node("research-agent", agent)
workflow.add_node("tools", ToolNode(tools))

# Build graph
workflow.add_edge(START, "retrieve")
workflow.add_conditional_edges("retrieve", grade_documents, {
    "transform": "rewrite",
    "unavailable": "research-agent",
    "available": "generate",
})
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("generate", "research-agent")
workflow.add_conditional_edges(
    "research-agent",
    tools_condition,
    {
        "tools": "tools",
    }
)
workflow.add_edge("tools", "research-agent")

# Compile
app = workflow.compile()


In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
from pprint import pprint

# Run
config = {"configurable": {"thread_id": "rag_thread", "user_id": "6746dd846f6d7f728d6cfba7"}}
inputs = "What the weather today at DANANG city"
msg = app.invoke({"messages": [("human", inputs)]}, config=config)

In [ ]:
for m in msg["messages"]:
    pprint(m.content)
    print('\n')

In [ ]:
from pprint import pprint

# Run
config = {"configurable": {"thread_id": "rag_thread", "user_id": "6746dd846f6d7f728d6cfba7"}}
inputs = "Base on my financial report of November 2024. Can you tell how money I have left to invest APPLE stock? Do you think APPLE can make profit for a long-term"
msg = app.invoke({"messages": [("human", inputs)]}, config=config)
